# Finance Strategy Expert — RAG Lab (Phase 1 prototype)

Interactive notebook to build and test the RAG pipeline **before** extracting modules.

**Stack (no extra signups, ~zero cost):**
- **OpenAI** for embeddings (`text-embedding-3-small`) and generation (`gpt-4o-mini`) — uses your existing OpenAI key.
- **Qdrant in-memory** (`QdrantClient(":memory:")`) — no Docker needed.
- Corpus: `data/corpus/*.md` (6 finance docs).

**Grounding rule:** answer ONLY from retrieved corpus chunks; if nothing passes the similarity threshold, reply *"I don't know."*

## 1. Install dependencies
Run once. Safe to skip if already installed. Uses **uv** (installs into this kernel's interpreter).

In [ ]:
import sys
!uv pip install -q --python {sys.executable} openai qdrant-client python-dotenv

## 2. Configuration & OpenAI key
Loads `.env` if present; otherwise prompts you to paste the key (not stored to disk).

In [ ]:
import os, glob, getpass
from dataclasses import dataclass
from pathlib import Path

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key (sk-...): ")

# Models
EMBED_MODEL = "text-embedding-3-small"   # 1536-dim, very cheap
CHAT_MODEL  = "gpt-4o-mini"              # cheap, good for grounded Q&A
VECTOR_SIZE = 1536

# Tunables (tweak these live in section 8)
TOP_K           = 5
SCORE_THRESHOLD = 0.30     # cosine; start permissive, tune later
CHUNK_WORDS     = 250

COLLECTION = "finance_kb"

# Locate the corpus whether the notebook runs from repo root or notebooks/
CORPUS_DIR = Path("data/corpus")
if not CORPUS_DIR.exists():
    CORPUS_DIR = Path("../data/corpus")
assert CORPUS_DIR.exists(), f"corpus not found at {CORPUS_DIR.resolve()}"

IDK_MESSAGE = "I don't know — I couldn't find that in my knowledge base."
SYSTEM_PROMPT = (
    "You are a strategy-investment expert. Answer the question using ONLY the "
    "context provided below. If the context does not contain the answer, reply "
    'exactly "I don\'t know." Do not use any outside knowledge. '
    "Cite the source title for each claim."
)
print("Corpus:", CORPUS_DIR.resolve())
print("Files:", len(list(CORPUS_DIR.glob("*.md"))))

## 3. Chunker — frontmatter parsing + word-bounded splitting

In [ ]:
@dataclass
class Chunk:
    text: str
    title: str
    source_url: str
    score: float | None = None


def parse_frontmatter(raw: str):
    """Return (meta dict, body) from a markdown doc with --- frontmatter."""
    if raw.startswith("---"):
        end = raw.index("\n---", 3)
        fm_block = raw[3:end].strip()
        body = raw[end + 4:].strip()
        meta = {}
        for line in fm_block.splitlines():
            if ":" in line:
                k, v = line.split(":", 1)
                meta[k.strip()] = v.strip().strip('"')
        return meta, body
    return {}, raw.strip()


def chunk_text(body: str, size: int):
    words = body.split()
    if not words:
        return []
    return [" ".join(words[i:i + size]) for i in range(0, len(words), size)]


def chunk_document(raw: str, size: int):
    meta, body = parse_frontmatter(raw)
    title = meta.get("title", "")
    url = meta.get("source_url", "")
    return [Chunk(text=t, title=title, source_url=url) for t in chunk_text(body, size)]


# Load + chunk the whole corpus
all_chunks = []
for path in sorted(CORPUS_DIR.glob("*.md")):
    all_chunks.extend(chunk_document(path.read_text(encoding="utf-8"), CHUNK_WORDS))

print(f"{len(all_chunks)} chunks from {len(list(CORPUS_DIR.glob('*.md')))} docs")
print("Example chunk:\n", all_chunks[0].title, "|", all_chunks[0].source_url)
print(all_chunks[0].text[:300], "...")

## 4. Embeddings (OpenAI)

In [ ]:
from openai import OpenAI

client = OpenAI()  # reads OPENAI_API_KEY from env


def embed(texts, _model=EMBED_MODEL):
    """Return a list of embedding vectors for a list of strings."""
    resp = client.embeddings.create(model=_model, input=texts)
    return [d.embedding for d in resp.data]


# quick sanity check
_v = embed(["hello world"])
print("vector dim:", len(_v[0]))

## 5. Ingest into in-memory Qdrant

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

qdrant = QdrantClient(":memory:")   # ephemeral; rebuilt each kernel session


def build_index(chunks):
    qdrant.recreate_collection(
        collection_name=COLLECTION,
        vectors_config=VectorParams(size=VECTOR_SIZE, distance=Distance.COSINE),
    )
    vectors = embed([c.text for c in chunks])
    points = [
        PointStruct(id=i, vector=vec,
                    payload={"text": c.text, "title": c.title, "source_url": c.source_url})
        for i, (c, vec) in enumerate(zip(chunks, vectors))
    ]
    qdrant.upsert(collection_name=COLLECTION, points=points)
    return len(points)


n = build_index(all_chunks)
print(f"Indexed {n} chunks into '{COLLECTION}'.")

## 6. Retriever — search + similarity threshold (the grounding gate)

In [ ]:
def search(question, top_k=None, threshold=None):
    top_k = TOP_K if top_k is None else top_k
    threshold = SCORE_THRESHOLD if threshold is None else threshold
    qvec = embed([question])[0]
    hits = qdrant.search(collection_name=COLLECTION, query_vector=qvec, limit=top_k)
    kept = []
    for h in hits:
        if h.score >= threshold:
            p = h.payload
            kept.append(Chunk(text=p["text"], title=p["title"],
                              source_url=p["source_url"], score=h.score))
    return kept


# Inspect what retrieval returns (including below-threshold, for tuning)
def debug_search(question, top_k=None):
    top_k = TOP_K if top_k is None else top_k
    qvec = embed([question])[0]
    hits = qdrant.search(collection_name=COLLECTION, query_vector=qvec, limit=top_k)
    print(f"Q: {question}\n")
    for h in hits:
        mark = "PASS" if h.score >= SCORE_THRESHOLD else "drop"
        print(f"  [{mark}] {h.score:.3f}  {h.payload['title']}")
        print(f"         {h.payload['text'][:120]}...\n")

debug_search("What is EV/EBIT?")

## 7. Answerer — grounded generation or "I don't know"
Short-circuits to *I don't know* when nothing passes the threshold (the LLM is never called).

In [ ]:
@dataclass
class Answer:
    text: str
    sources: list
    found: bool


def build_user_prompt(question, chunks):
    context = "\n\n".join(
        f"[Source: {c.title} | {c.source_url}]\n{c.text}" for c in chunks
    )
    return f"Context:\n{context}\n\nQuestion: {question}"


def answer(question):
    chunks = search(question)
    if not chunks:
        return Answer(text=IDK_MESSAGE, sources=[], found=False)
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_user_prompt(question, chunks)},
        ],
    )
    text = resp.choices[0].message.content
    sources = list(dict.fromkeys(f"{c.title} — {c.source_url}" for c in chunks))
    return Answer(text=text, sources=sources, found=True)


def ask(question):
    """Pretty-print an end-to-end answer."""
    r = answer(question)
    print(f"Q: {question}\n")
    print(r.text)
    if r.found and r.sources:
        print("\nSources:")
        for s in r.sources:
            print("  -", s)
    print("\n" + "-" * 60)
    return r

## 8. Test it 🎯

**In-corpus questions** (should answer with citations):

In [ ]:
ask("What is EV/EBIT and why do value investors use it?")
ask("What is the Acquirer's Multiple?")
ask("What did Tobias Carlisle find about mechanical value investing in Deep Value?")
ask("What is O'Shaughnessy's composite value factor?")

**Out-of-corpus question** (should say *I don't know*):

In [ ]:
ask("What's the weather in Tel Aviv today?")
ask("Who won the 2018 World Cup?")

## 9. Tune the grounding threshold
Use `debug_search` to see scores, then adjust `SCORE_THRESHOLD` and re-`ask`.
- Too many "I don't know" on valid questions → **lower** the threshold.
- Off-topic questions getting answered → **raise** it.
Once happy, copy `SCORE_THRESHOLD`, `TOP_K`, `CHUNK_WORDS` into the modules later.

In [ ]:
# Example: see raw scores for a borderline query, then experiment
debug_search("How does enterprise value differ from market cap?")

# Try a stricter threshold live:
# SCORE_THRESHOLD = 0.40
# ask("What's the weather in Tel Aviv today?")   # should now reliably say I don't know